# 09 - Stress Testing (mild + severe recession)

**Plain-English summary (read this first)**

A scorecard tells you the loss you expect in *normal* times. Regulators (Basel CRE36.51; the stress-testing
framework; APS 220) also require you to ask: **what happens to losses in a downturn?** This notebook applies
**recession scenarios** to the three loss drivers - **PD**, **LGD** and **EAD** - and recomputes portfolio
**Expected Loss (EL = PD x LGD x EAD)** under each, against a baseline.

It is deliberately simple and reproducible from the committed aggregate outputs (the calibrated grade PDs
in `14_pd_moc_overlay.csv` and the EAD/LGD figures in `ead_summary.csv`) - **no raw data needed**.

> **Scope note.** Like notebooks 08, these are **illustrative** figures: PD comes from the application
> scorecard (a competition-proxy default), LGD is an external-benchmark assumption, and EAD is a card-segment
> demonstration on a dataset whose default flag is known to be unreliable. The **method** is the deliverable;
> the numbers are not a capital plan.

## 1. Scenarios

| Scenario | Narrative | PD mult. | LGD mult. | EAD mult. |
|---|---|---|---|---|
| **Baseline** | Current calibrated through-the-door conditions | x1.00 | x1.00 | x1.00 |
| **Mild recession** | Basel's example downturn: ~two consecutive quarters of ~zero GDP growth, modest unemployment rise | x1.50 | x1.10 | x1.05 |
| **Severe recession** | Deep, prolonged contraction; sharp unemployment rise and asset-price falls | x2.50 | x1.25 | x1.10 |

**Why PD moves most:** in a downturn the *frequency* of default rises fastest; **LGD** rises more modestly
(recoveries weaken) and **EAD** a little (borrowers draw down available credit on the way into stress -
positive default/EAD correlation). The multipliers are **illustrative, documented** stresses, not estimated
from this single-snapshot dataset.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "output" / "scorecard_outputs" / "14_pd_moc_overlay.csv").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
SC = ROOT / "output" / "scorecard_outputs"
OUT_DIR = SC; print("Reading aggregates from:", SC)

# Calibrated, MoC-loaded grade PDs and grade sizes (PD-3 output).
grades = pd.read_csv(SC / "14_pd_moc_overlay.csv")            # grade, n, pd_post_moc, ...
# EAD (onset, primary) and benchmarked LGD (notebook 08 output).
ead = pd.read_csv(ROOT / "output" / "ead_summary.csv").set_index("metric")["value"]
EAD_BASE = float(ead["EAD_onset_primary_avg"])
LGD_BASE = float(ead["LGD_base"])
print(f"Baseline drivers: count-weighted PD={np.average(grades.pd_post_moc, weights=grades.n):.4f}  "
      f"LGD={LGD_BASE:.2f}  EAD={EAD_BASE:,.0f}")

Reading aggregates from: D:\Jane\Job Search\Github\bank\github project\scorecard pd ead consummer credit\output\scorecard_outputs
Baseline drivers: count-weighted PD=0.1018  LGD=0.75  EAD=2,005


In [2]:
# 2. Scenario multipliers and the EL engine.
scenarios = pd.DataFrame({
    "scenario":  ["Baseline", "Mild recession", "Severe recession"],
    "pd_mult":   [1.00, 1.50, 2.50],
    "lgd_mult":  [1.00, 1.10, 1.25],
    "ead_mult":  [1.00, 1.05, 1.10],
})

def portfolio_el(pd_mult, lgd_mult, ead_mult):
    # No-diversification convention (APG 113 para 92): the three stresses are applied SIMULTANEOUSLY
    # with no offsetting correlation benefit, and grade ELs are summed with no diversification credit.
    pd_g  = np.minimum(grades.pd_post_moc * pd_mult, 1.0)        # PD capped at 1.0
    lgd   = min(LGD_BASE * lgd_mult, 1.0)                        # LGD capped at 1.0
    ead   = EAD_BASE * ead_mult
    el_g  = grades.n * pd_g * lgd * ead                          # EL per grade = n x PD x LGD x EAD
    return float(el_g.sum())

scenarios["portfolio_EL"] = scenarios.apply(
    lambda r: portfolio_el(r.pd_mult, r.lgd_mult, r.ead_mult), axis=1)
base_el = scenarios.loc[scenarios.scenario == "Baseline", "portfolio_EL"].iloc[0]
scenarios["EL_uplift_vs_baseline"] = scenarios.portfolio_EL / base_el - 1.0
scenarios_disp = scenarios.copy()
scenarios_disp["portfolio_EL"] = scenarios_disp.portfolio_EL.round(0)
scenarios_disp["EL_uplift_vs_baseline"] = (scenarios_disp.EL_uplift_vs_baseline * 100).round(0).astype(int).astype(str) + "%"
print("Baseline-vs-stressed portfolio Expected Loss (illustrative):")
scenarios_disp

Baseline-vs-stressed portfolio Expected Loss (illustrative):


,scenario,pd_mult,lgd_mult,ead_mult,portfolio_EL,EL_uplift_vs_baseline
0,Baseline,1.0,1.00,1.00,14118482.0,0%
1,Mild recession,1.5,1.10,1.05,24460270.0,73%
2,Severe recession,2.5,1.25,1.10,48532282.0,244%


In [3]:
# 3. Save the stress result and print a plain-English read.
scenarios.round(4).to_csv(SC / "15_stress_test.csv", index=False)
print("Saved -> outputs/tables/scorecard_outputs/15_stress_test.csv\n")
mild   = scenarios[scenarios.scenario == "Mild recession"].iloc[0]
severe = scenarios[scenarios.scenario == "Severe recession"].iloc[0]
print("Plain-English read:")
print(f"  - A MILD recession lifts portfolio EL by ~{mild.EL_uplift_vs_baseline*100:.0f}% vs baseline,")
print(f"    driven mostly by the PD x1.5 frequency effect.")
print(f"  - A SEVERE recession lifts EL by ~{severe.EL_uplift_vs_baseline*100:.0f}% - the kind of loss")
print(f"    a downturn capital buffer must absorb. The gap between this and baseline is the")
print(f"    'cyclical' portion of capital the stress test exists to size.")

Saved -> outputs/tables/scorecard_outputs/15_stress_test.csv

Plain-English read:
  - A MILD recession lifts portfolio EL by ~73% vs baseline,
    driven mostly by the PD x1.5 frequency effect.
  - A SEVERE recession lifts EL by ~244% - the kind of loss
    a downturn capital buffer must absorb. The gap between this and baseline is the
    'cyclical' portion of capital the stress test exists to size.


## 4. Framework notes

- **No-diversification conservatism (APG 113 para 92).** The PD, LGD and EAD stresses are applied
  **simultaneously** and grade ELs are **summed with no diversification credit**. This is deliberately
  conservative - a real economic-capital model might recognise some diversification, but for a stress
  *floor* we assume none.

- **Management actions / contingency (APS 220 para 74).** In a real downturn the loss above is **before**
  management response. A bank would document contingent actions - tightening cut-offs, reducing limits on
  high-utilisation accounts, repricing, raising provisions early, pausing originations in stressed segments -
  and show the EL **net** of those actions. Here we show the gross (pre-action) stress only.

- **Reverse stress test (framing).** The mirror-image question: *how severe a scenario would it take to
  exhaust the capital/provision buffer?* One would solve for the PD/LGD/EAD multipliers that push EL to the
  buffer limit, then judge whether that scenario is plausible. Not computed here (no capital figure on a
  demo book), but it is the natural next step.

- **Independent validation (APS 220 para 76).** A production stress framework - scenarios, multipliers and
  the EL engine - would be **independently validated** (by a function separate from model development) and
  approved through model risk governance. In this repo development and validation are separated only by notebook.

- **Where a fuller stress test lives.** The companion **Freddie Mac mortgage** project runs the same idea on
  a clean monthly performance panel with real default/termination outcomes.